# Feature Research Lab

Score a feature's **predictive signal** before building a model.

**Two ways to run, same notebook:**
- **Local kernel** (your IDE's Colab extension on the repo venv): can regenerate the panel from live OANDA data via `research.feature_lab`.
- **Cloud Colab / sharing with Gemini:** just upload `data/research/feature_panel_swing.parquet` and run — pure pandas, no repo or TA-Lib needed.

**What to read:** a feature has signal if its **IC** (rank-correlation with the future move) is steady over time, its **decile staircase** climbs, it **survives out-of-sample**, and it **beats cost** — and clearly outscores the built-in `noise_random` control.


## 1 · Load the panel + report


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from scipy import stats

PROFILE = 'swing'
PANEL = f'../data/research/feature_panel_{PROFILE}.parquet'   # adjust path in Colab
REPORT = f'../data/research/feature_report_{PROFILE}.csv'

panel = pd.read_parquet(PANEL)
report = pd.read_csv(REPORT)
panel['timestamp'] = pd.to_datetime(panel['timestamp'], utc=True)

META = {'timestamp','symbol','open','high','low','close','volume',
        'fwd_return','angel_target','devil_target'}
FEATURES = [c for c in panel.columns if c not in META]
OOS_SPLIT = panel['timestamp'].max() - pd.Timedelta(days=365)
print(f'{len(panel):,} rows | {len(FEATURES)} features | OOS from {OOS_SPLIT.date()}')
report.head(8)


## 2 · Portable scoring helpers (pure pandas — work in cloud Colab)


In [ ]:
def ic(df, feat, target='fwd_return'):
    d = df[[feat, target]].dropna()
    if len(d) < 30: return np.nan
    return stats.spearmanr(d[feat], d[target]).correlation

def decile_table(df, feat, target='fwd_return', q=10):
    d = df[[feat, target]].dropna().copy()
    d['bucket'] = pd.qcut(d[feat].rank(method='first'), q, labels=False)
    return d.groupby('bucket')[target].mean() * 1e4   # basis points

def yearly_ic(df, feat, target='fwd_return'):
    d = df[['timestamp', feat, target]].dropna().copy()
    d['year'] = d['timestamp'].dt.year
    return d.groupby('year').apply(lambda g: stats.spearmanr(g[feat], g[target]).correlation)

# sanity: the noise control should be ~0
print('noise_random IC:', round(ic(panel, 'noise_random'), 4))


## 3 · Ranked information ratio (which features are *consistent*)


In [ ]:
r = report.sort_values('ic_ir_is', key=lambda s: s.abs(), ascending=True)
plt.figure(figsize=(8, 7))
colors = ['crimson' if f=='noise_random' else 'steelblue' for f in r['feature']]
plt.barh(r['feature'], r['ic_ir_is'], color=colors)
plt.axvline(0, color='k', lw=0.8)
plt.title('In-sample IC information ratio (red = noise control)')
plt.xlabel('mean IC / std IC'); plt.tight_layout(); plt.show()


## 4 · Decile staircase (does the top bucket actually move more?)


In [ ]:
FEATURE = 'natr_14'   # <- change to any feature
dt = decile_table(panel, FEATURE)
plt.figure(figsize=(7,4))
plt.bar(dt.index, dt.values, color='steelblue')
plt.axhline(0, color='k', lw=0.8)
plt.title(f'{FEATURE}: mean forward return by decile')
plt.xlabel('feature decile (low -> high)'); plt.ylabel('forward return (bps)')
plt.tight_layout(); plt.show()
print('top-minus-bottom spread (bps):', round(dt.iloc[-1]-dt.iloc[0], 2))


## 5 · Stability over time (sign flips = fragile)


In [ ]:
yi = yearly_ic(panel, FEATURE)
plt.figure(figsize=(7,4))
plt.bar(yi.index.astype(str), yi.values, color=['seagreen' if v>=0 else 'crimson' for v in yi.values])
plt.axhline(0, color='k', lw=0.8)
plt.title(f'{FEATURE}: IC by year'); plt.ylabel('IC'); plt.tight_layout(); plt.show()
yi


## 6 · Redundancy heatmap (don't add a copy of what you have)


In [ ]:
corr = panel[FEATURES].corr().abs()
plt.figure(figsize=(9,8))
plt.imshow(corr, cmap='magma', vmin=0, vmax=1)
plt.xticks(range(len(FEATURES)), FEATURES, rotation=90, fontsize=7)
plt.yticks(range(len(FEATURES)), FEATURES, fontsize=7)
plt.colorbar(label='|correlation|'); plt.title('feature redundancy'); plt.tight_layout(); plt.show()


## 7 · ⭐ Define YOUR candidate feature here

Write a function that adds one column to the panel, then score it against the targets **and** the noise control. 
This works in cloud Colab too (pure pandas). For features needing fresh indicators, regenerate the panel locally via `python -m research.feature_lab`.


In [ ]:
def add_candidate(df):
    # EXAMPLE: 3-bar momentum of RSI. Replace with your idea.
    df = df.sort_values(['symbol','timestamp']).copy()
    df['my_feature'] = df.groupby('symbol')['rsi_14'].diff(3)
    return df

p2 = add_candidate(panel)
is_, oos_ = p2[p2.timestamp < OOS_SPLIT], p2[p2.timestamp >= OOS_SPLIT]
print('my_feature   IC full :', round(ic(p2,'my_feature'), 4))
print('my_feature   IC OOS  :', round(ic(oos_,'my_feature'), 4))
print('my_feature   vs angel:', round(ic(p2,'my_feature','angel_target'), 4))
print('noise_random IC full :', round(ic(p2,'noise_random'), 4), ' <- must stay ~0')
decile_table(p2, 'my_feature')
